<a href="https://colab.research.google.com/github/Josiah-Kunz/MGN-Public/blob/main/examples/colab/2_meshgraphnet/1_train_mgn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Example of applying a mesh graph network (MGN).

MGN uses FEM context directly (boundaries, loads, mesh topology) rather than CSV files.
This allows it to learn geometry-agnostic physics that can transfer to new geometries.

In [1]:
%%capture

try:
    import google.colab  # noqa: F401
except ImportError:
    import ufl_legacy
    import fenics
else:
    try:
        import ufl_legacy
        import fenics
    except ImportError:
        !wget "https://fem-on-colab.github.io/releases/fenics-install-release-real.sh" -O "/tmp/fenics-install.sh" && bash "/tmp/fenics-install.sh"
        import ufl_legacy
        import fenics

!apt-get install -y libglu1-mesa -q
!pip install gmsh meshio pint torch torch-geometric scikit-learn -q
!pip install meshgraphnet -q

import os
from meshgraphnet import *
import numpy as np

In [2]:
# Settings

MESH_SIZE = 25 # mm

# Get the geometry to train on
train_geometries = [
    "plate-with-hole_8in.stp",
    "plate-with-hole_8in_square.stp",
    "plate-with-hole_8in_ellipse.stp",
    "plate-with-hole_4in.stp",
    "plate-with-hole_8in_double.stp",
    "plate-with-hole_8in_offset-right.stp",
    "plate-with-hole_8in_offset-right_1in.stp",
    ]
for geometry in train_geometries:
  !wget "https://raw.githubusercontent.com/Josiah-Kunz/MGN-Public/refs/heads/main/examples/local/plate_with_hole/geometries/{geometry}" -O "{geometry}" -q


# Train loads
num_loads = 20
TRAIN_LOADS = (
        np.linspace(1000, 4500, num_loads//2, dtype=int).tolist() +
        np.linspace(5500, 10000, num_loads//2, dtype=int).tolist()
)

# Test geometry + load
# (just to make sure we're okay; more rigorous testing in the next example)
TEST_GEOMETRY = "plate-with-hole_8in_hex.stp"
!wget "https://raw.githubusercontent.com/Josiah-Kunz/MGN-Public/refs/heads/main/examples/local/plate_with_hole/geometries/{TEST_GEOMETRY}" -O "{TEST_GEOMETRY}" -q
TEST_LOAD = 5000

In [3]:
def main():

  # Create visualizations
    mgn = MGN(
        hidden_channels=64,
        num_layers=20,
        global_features=["load"],
        learning_rate=1e-4,
        epochs=5000,
    )

    # Create training FEMs (all geometries x all loads)
    train_fems = []
    for geom_file in train_geometries:
        mesh = MeshObject(geom_file, MESH_SIZE, units=Units.SI_MM, force_2d=True, convert_to=Units.US_IN)
        print(f"\nGeometry: {geom_file} ({len(mesh.dolfin_mesh.coordinates())} nodes)")

        for load in TRAIN_LOADS:
            fem = create_fem(mesh, load, geom_file)
            fem.solve()
            train_fems.append(fem)

        print(f"\tSolved {len(TRAIN_LOADS)} load cases")

    print(f"\nTotal training FEMs: {len(train_fems)}")

    # Create test FEM
    test_mesh = MeshObject(TEST_GEOMETRY, MESH_SIZE, units=Units.SI_MM, force_2d=True, convert_to=Units.US_IN)
    test_fem = create_fem(test_mesh, TEST_LOAD, TEST_GEOMETRY)
    test_fem.solve()
    print(f"\nTest geometry: {TEST_GEOMETRY} ({len(test_mesh.dolfin_mesh.coordinates())} nodes)")

    # Define ML wrapper
    ml = MLObject(
        train_fem=train_fems,
        test_fem=test_fem,
        objectives=["von_mises"],
        name="Multi-Geometry MGN"
    )

    # Train MGN
    mgn_filename = "results/multi_geometry_mgn.pt"
    ml.train(model=mgn, model_name="MGN", batch_size=1)
    mgn.save(mgn_filename)

    # Evaluate on unseen geometry and unseen load
    ml.evaluate_on_unseen_data()
    ml.plot_predictions()
    ml.plot_loss()
    ml.plot_ml_vs_fem(test_mesh)

    # Test on each training geometry with unseen load
    print("\n" + "="*50)
    print("Evaluating on training geometries with unseen load:")
    print("="*50)
    for geom_file in train_geometries:
        mesh = MeshObject(geom_file, MESH_SIZE, units=Units.SI_MM, force_2d=True, convert_to=Units.US_IN)
        fem = create_fem(mesh, TEST_LOAD, geom_file)  # Unseen load
        fem.solve()
        r2 = mgn.score(fem, fem.von_mises)
        print(f"  {geom_file}: R² = {r2:.4f}")

In [4]:
def create_fem(mesh, load, geom_name):
    """Create FEM object for a given mesh and load (but does *not* solve it)."""
    material = Material.steel()

    loads = LoadCollection(units=Units.US_IN)
    loads.add(SurfaceLoad(
        location=lambda x, on_boundary: on_boundary and x[0] > mesh.bounds["x"][1] - 1e-6,
        pressure=(load, 0, 0),
        name="Applied Load"
    ))

    fem = FEMObject(
        mesh=mesh,
        loads=loads,
        boundaries=[FixedBoundary('left', value=(0, 0, 0), name="Fixed")],
        material=material,
        name=geom_name.replace('.stp', '').replace('-', '_'),
        metadata={"load": load},
    )
    return fem

## Actual Training Not Recommended on Colab
Colab tends to use either CPU or poorly performing GPU. Consequently, training takes a long time (~10 hours, as seen in the next cell). It is recommended to train locally instead.

image.png


In [5]:
if __name__ == "__main__":
    main()

MGN using: cuda

Geometry: plate-with-hole_8in.stp (788 nodes)
	Solved 20 load cases

Geometry: plate-with-hole_8in_square.stp (772 nodes)
	Solved 20 load cases

Geometry: plate-with-hole_8in_ellipse.stp (813 nodes)
	Solved 20 load cases

Geometry: plate-with-hole_4in.stp (838 nodes)
	Solved 20 load cases

Geometry: plate-with-hole_8in_double.stp (749 nodes)
	Solved 20 load cases

Geometry: plate-with-hole_8in_offset-right.stp (792 nodes)
	Solved 20 load cases

Geometry: plate-with-hole_8in_offset-right_1in.stp (786 nodes)
	Solved 20 load cases

Total training FEMs: 140

Test geometry: plate-with-hole_8in_hex.stp (776 nodes)
Node types found:
  applied_load: 1680
  fixed: 1680
  free: 16800
  hole: 4000
  interior: 86600
  Total: 110760


/usr/local/lib/python3.12/dist-packages/meshgraphnet/ml_object/mgn.py:312: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler() if self.device.type == 'cuda' else None
Training MGN:   0%|          | 0/5000 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/meshgraphnet/ml_object/mgn.py:399: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/usr/local/lib/python3.12/dist-packages/meshgraphnet/ml_object/mgn_model.py:130: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
Training MGN:   0%|          | 3/5000 [00:20<9:22:12,  6.75s/it, loss=0.4858]


KeyboardInterrupt: 